In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from src.utils import load_config
from src.dataset import open_zarr, load_subset_indices, load_features

cfg = load_config("../configs/config.yaml")
print("Config loaded")

I0000 00:00:1779436083.461139   58481 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Config loaded


In [2]:
store = open_zarr(cfg["data"]["raw_path"])
idx = load_subset_indices(store, cfg)

feats = load_features(
    store,
    idx,
    cfg,
    neighbors_npz_path="/group/sottoriva/andrey.tyshevich/Foundational_Spatial_Model/data/cache/processed_neighbors_same_patient_same_sample.npz"
)
print("Dati caricati")

09:48:29 | INFO | dataset | Zarr aperto: /group/sottoriva/andrey.tyshevich/Foundational_Spatial_Model/foundational_model_data.zarr/
/home/matteo.doria/Desktop/ASLCD/.venv/lib/python3.12/site-packages/zarr/core/group.py:3559: ZarrUserWarning: Object at .DS_Store is not recognized as a component of a Zarr hierarchy.
  warnings.warn(
09:48:29 | INFO | dataset | Chiavi: ['neighbors_idxs', 'HE_embeddings', 'cell_perimeter', 'neighbors_radius', 'cell_extent', 'cell_eccentricity', 'positions', 'neighbors_angle', 'cell_area', 'expression', 'sample', 'cell_types', 'patient', 'idx_to_take_into_training_based_on_grid']
09:48:29 | INFO | dataset | N celle totali: 8,065,561
09:48:29 | INFO | dataset | Subset: 100,000 / 8,065,561 celle
09:48:29 | INFO | dataset | Caricamento feature dal zarr ...
09:52:58 | INFO | dataset | Caricamento neighbors corretti da cache: /group/sottoriva/andrey.tyshevich/Foundational_Spatial_Model/data/cache/processed_neighbors_same_patient_same_sample.npz
09:53:09 | INFO |

Dati caricati


In [4]:
# 1. HE → PCA 64
pca = PCA(n_components=64, random_state=42)
he_pca = pca.fit_transform(feats["he"].astype(np.float32))
print(f"HE PCA: {he_pca.shape} — varianza spiegata: {pca.explained_variance_ratio_.sum():.3f}")

# 2. Morfologia normalizzata
scaler = StandardScaler()
morph_norm = scaler.fit_transform(feats["morph"]).astype(np.float32)

# 3. Cell types one-hot
cell_type_ids = feats["cell_types"].argmax(axis=1).astype(np.int8)
n_types = int(cell_type_ids.max() + 1)
celltype_oh = np.eye(n_types, dtype=np.float32)[cell_type_ids]

# 4. Feature tabellari complete (76 dim)
X_num = np.concatenate([he_pca, morph_norm], axis=1).astype(np.float32)
Xtab = np.concatenate([X_num, celltype_oh], axis=1).astype(np.float32)
print(f"Xtab: {Xtab.shape}")

# 5. Target: top-2000 HVG
def prepare_target(rna, hvg_path, topk=2000):
    hvg_df = pd.read_csv(hvg_path, sep="\t")
    hvg_df = hvg_df.rename(columns={"Unnamed: 0": "gene_name"})
    top_hvg = hvg_df.nsmallest(topk, "highly_variable_rank")
    hvg_idx = top_hvg.index.tolist()
    return rna[:, hvg_idx].astype(np.float32), top_hvg["gene_name"].tolist()

rna_hvg, gene_names = prepare_target(feats["rna"], cfg["data"]["hvg_path"])
print(f"Target RNA: {rna_hvg.shape}")

HE PCA: (100000, 64) — varianza spiegata: 0.712
Xtab: (100000, 76)
Target RNA: (100000, 2000)


In [6]:
def remap_nn_ids(nnids, global_idx, padding=-1):
    global_to_local = {int(g): i for i, g in enumerate(global_idx)}
    flat = nnids.ravel()
    mapped = np.full_like(flat, fill_value=padding, dtype=np.int32)
    for i, g in enumerate(flat):
        if g == padding:
            continue
        mapped[i] = global_to_local.get(int(g), padding)
    return mapped.reshape(nnids.shape)

nn_local = remap_nn_ids(feats["nn_ids"], idx)
print(f"nn_local: {nn_local.shape} — padding: {(nn_local == -1).sum()}")

nn_local: (100000, 8) — padding: 790156


In [7]:
def aggregate_neighbors(X, nn_local, n_feats=76):
    N, K = nn_local.shape
    X_nbr = np.zeros((N, n_feats), dtype=np.float32)
    counts = np.zeros(N, dtype=np.float32)
    for k in range(K):
        valid = nn_local[:, k] != -1
        X_nbr[valid] += X[nn_local[valid, k]]
        counts[valid] += 1
    counts = np.maximum(counts, 1)
    X_nbr /= counts[:, None]
    return X_nbr

X_nbr = aggregate_neighbors(Xtab, nn_local)
print(f"X_nbr: {X_nbr.shape}")

X_nbr: (100000, 76)


In [8]:
train_mask = feats["train_mask"].astype(bool)

X_train, X_test   = Xtab[train_mask], Xtab[~train_mask]
X_nbr_train, X_nbr_test = X_nbr[train_mask], X_nbr[~train_mask]
y_train, y_test   = rna_hvg[train_mask], rna_hvg[~train_mask]

X_tr, X_val, X_nbr_tr, X_nbr_val, y_tr, y_val = train_test_split(
    X_train, X_nbr_train, y_train,
    test_size=0.15, random_state=42, shuffle=True
)

print(f"Train:  {X_tr.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train:  (67409, 76), Val: (11896, 76), Test: (20695, 76)


In [9]:
class MeanNeighborLayer(layers.Layer):
    def call(self, inputs):
        X, nn_local = inputs
        batch_size = tf.shape(X)[0]
        n_feats = tf.shape(X)[1]
        k = tf.shape(nn_local)[1]

        nn_clipped = tf.maximum(nn_local, 0)
        nbr_feats = tf.gather(X, nn_clipped)

        mask = tf.cast(tf.not_equal(nn_local, -1), tf.float32)
        mask = tf.expand_dims(mask, axis=-1)

        nbr_sum = tf.reduce_sum(nbr_feats * mask, axis=1)
        nbr_count = tf.reduce_sum(mask, axis=1)
        nbr_count = tf.maximum(nbr_count, 1.0)

        return nbr_sum / nbr_count

cell_in = keras.Input(shape=(76,), name="cell_input")
nn_in = keras.Input(shape=(8,), dtype=tf.int32, name="nn_input")

nbr_agg = MeanNeighborLayer()([cell_in, nn_in])

x1 = layers.Dense(256, activation="relu")(cell_in)
x1 = layers.BatchNormalization()(x1)
x1 = layers.Dropout(0.3)(x1)

x2 = layers.Dense(256, activation="relu")(nbr_agg)
x2 = layers.BatchNormalization()(x2)
x2 = layers.Dropout(0.3)(x2)

x = layers.Concatenate()([x1, x2])
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
out = layers.Dense(2000, activation="linear")(x)

model_gnn = keras.Model([cell_in, nn_in], out)
model_gnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
model_gnn.summary()

E0000 00:00:1779437270.826763   58481 cuda_executor.cc:1827] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
E0000 00:00:1779437274.471764   58481 cuda_executor.cc:1827] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
E0000 00:00:1779437274.471977   58481 cuda_executor.cc:1827] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
E0000 00:00:1779437274.473315   58481 cuda_executor.cc:1827] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
E0000 00:00:1779437274.473514   58481 cuda_executor.cc:1827] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
E0000 00:00:1779437274.473676   58481 cuda_executor.cc:1827] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
E0000 00:00:1779437274.627656   58481 cuda_executor.cc:1827] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
E0000 00:00:1779437274.627878   58481 cuda_execu

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cell_input          │ (None, 76)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nn_input            │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mean_neighbor_layer │ (None, 76)        │          0 │ cell_input[0][0], │
│ (MeanNeighborLayer) │                   │            │ nn_input[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     19,712 │ cell_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     19,712 │ mean_neighbor_la… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │    131,328 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     32,896 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 2000)      │    258,000 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 463,696 (1.77 MB)

 Trainable params: 462,672 (1.76 MB)

 Non-trainable params: 1,024 (4.00 KB)

In [10]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    ),
]

history_gnn = model_gnn.fit(
    [X_tr, nn_local[train_mask][np.arange(len(X_tr))]],
    y_tr,
    validation_data=(
        [X_val, nn_local[train_mask][len(X_tr):len(X_tr)+len(X_val)]],
        y_val
    ),
    epochs=30,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30


I0000 00:00:1779437296.563962   58833 service.cc:153] XLA service 0x7fe41c059c80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779437296.563992   58833 service.cc:161]   StreamExecutor [0]: NVIDIA A40-24Q, Compute Capability 8.6 (Driver: 13.0.0; Runtime: 12.6.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1779437297.202817   58833 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779437298.048315   58833 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1779437298.207292   58833 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3129__.27
I0000 00:00:1779437298.374532   58833 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:

 56/264 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0295 - mae: 0.0928

I0000 00:00:1779437316.020403   58833 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


247/264 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0168 - mae: 0.0488

I0000 00:00:1779437317.147319   58831 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3129__.27
I0000 00:00:1779437317.236386   58831 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1779437320.400248   58831 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1779437323.094301   58831 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1779437326.879799

264/264 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - loss: 0.0165 - mae: 0.0476

I0000 00:00:1779437342.376997   58832 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1779437344.946834   58832 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


264/264 ━━━━━━━━━━━━━━━━━━━━ 56s 131ms/step - loss: 0.0116 - mae: 0.0291 - val_loss: 0.0099 - val_mae: 0.0212 - learning_rate: 0.0010
Epoch 2/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0098 - mae: 0.0217 - val_loss: 0.0096 - val_mae: 0.0212 - learning_rate: 0.0010
Epoch 3/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0096 - mae: 0.0215 - val_loss: 0.0095 - val_mae: 0.0213 - learning_rate: 0.0010
Epoch 4/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0094 - mae: 0.0216 - val_loss: 0.0092 - val_mae: 0.0217 - learning_rate: 0.0010
Epoch 5/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0089 - mae: 0.0215 - val_loss: 0.0086 - val_mae: 0.0215 - learning_rate: 0.0010
Epoch 6/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0086 - mae: 0.0214 - val_loss: 0.0085 - val_mae: 0.0213 - learning_rate: 0.0010
Epoch 7/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0085 - mae: 0.0213 - val_loss: 0.0084 - val_mae: 0.0213 - learning_rate: 0.0010
Epoch 8/30
264

In [11]:
# pesi vicini
if "nn_radius" in feats:
    nbr_r = feats["nn_radius"].astype(np.float32)
    w_all = 1.0 / (nbr_r + 1e-3)
    w_all[nn_local == -1] = 0.0
else:
    ranks = np.arange(1, nn_local.shape[1] + 1, dtype=np.float32)[None, :]
    w_all = np.repeat(1.0 / ranks, nn_local.shape[0], axis=0)
    w_all[nn_local == -1] = 0.0

w_sum = w_all.sum(axis=1, keepdims=True)
w_sum[w_sum == 0] = 1.0
w_all = w_all / w_sum

# aggregazione pesata dei vicini
X_nbr_w = np.zeros_like(Xtab, dtype=np.float32)
for k in range(nn_local.shape[1]):
    valid = nn_local[:, k] != -1
    X_nbr_w[valid] += Xtab[nn_local[valid, k]] * w_all[valid, k:k+1]

# split coerente
X_train, X_test = Xtab[train_mask], Xtab[~train_mask]
X_nbr_train, X_nbr_test = X_nbr_w[train_mask], X_nbr_w[~train_mask]
y_train, y_test = rna_hvg[train_mask], rna_hvg[~train_mask]

X_tr, X_val, X_nbr_tr, X_nbr_val, y_tr, y_val = train_test_split(
    X_train, X_nbr_train, y_train,
    test_size=0.15, random_state=42, shuffle=True
)

# modello
cell_in = keras.Input(shape=(76,), name="cell_input")
nbr_in = keras.Input(shape=(76,), name="neighbor_input")

x1 = layers.Dense(256, activation="relu")(cell_in)
x1 = layers.BatchNormalization()(x1)
x1 = layers.Dropout(0.3)(x1)
x1 = layers.Dense(128, activation="relu")(x1)

x2 = layers.Dense(256, activation="relu")(nbr_in)
x2 = layers.BatchNormalization()(x2)
x2 = layers.Dropout(0.3)(x2)
x2 = layers.Dense(128, activation="relu")(x2)

x = layers.Concatenate()([x1, x2])
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
out = layers.Dense(2000, activation="linear")(x)

model_gnn_w = keras.Model([cell_in, nbr_in], out)
model_gnn_w.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    ),
]

history_gnn_w = model_gnn_w.fit(
    [X_tr, X_nbr_tr], y_tr,
    validation_data=([X_val, X_nbr_val], y_val),
    epochs=30,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

test_loss, test_mae = model_gnn_w.evaluate([X_test, X_nbr_test], y_test, verbose=1)
print("Test loss:", test_loss)
print("Test MAE:", test_mae)

Epoch 1/30


I0000 00:00:1779437570.821423   58833 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_46266__.27


253/264 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0131 - mae: 0.0330

I0000 00:00:1779437575.028120   58833 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_46266__.27
I0000 00:00:1779437575.352632   58833 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


264/264 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.0130 - mae: 0.0326

...

264/264 ━━━━━━━━━━━━━━━━━━━━ 19s 53ms/step - loss: 0.0107 - mae: 0.0238 - val_loss: 0.0099 - val_mae: 0.0213 - learning_rate: 0.0010
Epoch 2/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0098 - mae: 0.0213 - val_loss: 0.0096 - val_mae: 0.0214 - learning_rate: 0.0010
Epoch 3/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0096 - mae: 0.0213 - val_loss: 0.0095 - val_mae: 0.0214 - learning_rate: 0.0010
Epoch 4/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0094 - mae: 0.0214 - val_loss: 0.0094 - val_mae: 0.0214 - learning_rate: 0.0010
Epoch 5/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0094 - mae: 0.0214 - val_loss: 0.0094 - val_mae: 0.0214 - learning_rate: 0.0010
Epoch 6/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0094 - mae: 0.0214 - val_loss: 0.0093 - val_mae: 0.0214 - learning_rate: 0.0010
Epoch 7/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0093 - mae: 0.0214 - val_loss: 0.0093 - val_mae: 0.0214 - learning_rate: 0.0010
Epoch 8/30
264/

I0000 00:00:1779437619.230946   58833 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


633/647 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0094 - mae: 0.0210

I0000 00:00:1779437625.041510   58830 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


647/647 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.0092 - mae: 0.0210
Test loss: 0.009193101897835732
Test MAE: 0.021015550941228867
